In [0]:
%sql
SHOW TABLES IN ecommerce.silver;

database,tableName,isTemporary
silver,customer_address_default_exceptions,false
silver,customer_address_orphan_exceptions,false
silver,customer_addresses,false
silver,customer_addresses_external,false
silver,customer_exceptions,false
silver,customer_quality_metrics,false
silver,customer_quality_summary,false
silver,customer_quarantine,false
silver,customers,false
silver,customers_external,false


In [0]:
%sql

DESCRIBE TABLE ecommerce.silver.payments;

col_name,data_type,comment
payment_id,bigint,null
order_id,bigint,null
payment_method,string,null
transaction_id,string,null
payment_amount,"decimal(12,2)",null
payment_status,string,null
payment_date,timestamp,null
refund_amount,"decimal(12,2)",null
created_at,timestamp,null
updated_at,timestamp,null


In [0]:
%sql
SELECT *
FROM ecommerce.silver.order_item_product_exceptions
WHERE product_id = 3998;

order_item_id,order_id,product_id,quantity,unit_price,discount_amount,tax_amount,line_total,created_at,updated_at,exception_reason,exception_detected_at
225,45829,3998,5,52222.14,1874.73,46662.47,305898.44,2024-07-24T12:56:35.000Z,2024-07-26T12:56:35.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
14091,68593,3998,1,52222.14,3517.02,8766.92,57472.04,2024-09-03T14:38:51.000Z,2024-10-03T14:38:51.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
17685,11484,3998,4,52222.14,34463.22,31396.56,205821.90,2026-07-30T09:43:06.000Z,2026-08-14T09:43:06.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
21385,96196,3998,3,52222.14,1857.81,27865.55,182674.16,2024-12-27T06:08:13.000Z,2025-01-15T06:08:13.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
21498,98870,3998,4,52222.14,27489.89,32651.76,214050.43,2024-01-31T08:41:46.000Z,2024-02-04T08:41:46.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
34273,68412,3998,3,52222.14,24908.31,23716.46,155474.57,2026-08-25T05:32:55.000Z,2026-09-13T05:32:55.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
40549,16381,3998,1,52222.14,8296.24,7906.66,51832.56,2024-10-22T17:11:08.000Z,2024-10-27T17:11:08.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
41265,59772,3998,2,52222.14,12043.72,16632.10,109032.66,2025-11-16T02:05:49.000Z,2025-11-28T02:05:49.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
43428,74282,3998,5,52222.14,37795.20,40196.79,263512.29,2025-10-25T16:48:50.000Z,2025-10-28T16:48:50.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z
52840,27751,3998,5,52222.14,23440.77,42780.59,280450.52,2024-12-05T08:02:11.000Z,2024-12-24T08:02:11.000Z,MISSING_PRODUCT_MASTER,2026-10-03T08:58:14.074Z


In [0]:
%sql
SELECT
    COUNT(*) AS total_addresses,
    SUM(CASE WHEN is_effective_default THEN 1 ELSE 0 END)
        AS effective_default_addresses,
    COUNT(DISTINCT CASE
        WHEN is_effective_default THEN customer_id
    END) AS customers_with_effective_default,
    SUM(CASE
        WHEN customer_id IS NULL THEN 1 ELSE 0
    END) AS addresses_without_customer
FROM ecommerce.silver.customer_addresses;

total_addresses,effective_default_addresses,customers_with_effective_default,addresses_without_customer
30000,25001,25001,0


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.dim_customer
USING DELTA
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
    customer_id,
    first_name,
    last_name,
    CONCAT_WS(' ', first_name, last_name) AS full_name,
    email,
    phone,
    gender,
    date_of_birth,
    registration_date,
    customer_status,
    created_at,
    updated_at
FROM ecommerce.silver.customers;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    COUNT(DISTINCT customer_key) AS distinct_customer_keys
FROM ecommerce.gold.dim_customer;

total_rows,distinct_customers,distinct_customer_keys
25001,25001,25001


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.dim_product
USING DELTA
AS
SELECT
    product_id AS product_key,
    product_id,
    product_name,
    category_id,
    category_name,
    brand,
    price,
    cost,
    stock_quantity,
    product_status,
    created_at,
    updated_at
FROM ecommerce.silver.products

UNION ALL

SELECT
    0 AS product_key,
    CAST(NULL AS INT) AS product_id,
    'Unknown Product' AS product_name,
    CAST(NULL AS INT) AS category_id,
    'Unknown' AS category_name,
    'Unknown' AS brand,
    CAST(NULL AS DECIMAL(12,2)) AS price,
    CAST(NULL AS DECIMAL(12,2)) AS cost,
    CAST(NULL AS INT) AS stock_quantity,
    'UNKNOWN' AS product_status,
    CAST(NULL AS TIMESTAMP) AS created_at,
    CAST(NULL AS TIMESTAMP) AS updated_at;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    'orders' AS source_table,
    MIN(TO_DATE(order_date)) AS first_date,
    MAX(TO_DATE(order_date)) AS last_date
FROM ecommerce.silver.orders

UNION ALL

SELECT
    'payments' AS source_table,
    MIN(TO_DATE(payment_date)) AS first_date,
    MAX(TO_DATE(payment_date)) AS last_date
FROM ecommerce.silver.payments;

source_table,first_date,last_date
orders,2024-01-01,2026-09-29
payments,2024-01-01,2026-09-29


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.dim_date
USING DELTA
AS
WITH calendar AS (
    SELECT EXPLODE(
        SEQUENCE(
            DATE '2024-01-01',
            DATE '2026-09-29',
            INTERVAL 1 DAY
        )
    ) AS calendar_date
)
SELECT
    CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
    calendar_date,
    YEAR(calendar_date) AS year,
    QUARTER(calendar_date) AS quarter,
    MONTH(calendar_date) AS month_number,
    DATE_FORMAT(calendar_date, 'MMMM') AS month_name,
    DATE_FORMAT(calendar_date, 'yyyy-MM') AS year_month,
    DAY(calendar_date) AS day_of_month,
    DAYOFWEEK(calendar_date) AS day_of_week,
    DATE_FORMAT(calendar_date, 'EEEE') AS day_name,
    CASE
        WHEN DAYOFWEEK(calendar_date) IN (1, 7) THEN TRUE
        ELSE FALSE
    END AS is_weekend
FROM calendar;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_dates,
    MIN(calendar_date) AS first_date,
    MAX(calendar_date) AS last_date,
    COUNT(DISTINCT date_key) AS distinct_date_keys
FROM ecommerce.gold.dim_date;

total_dates,first_date,last_date,distinct_date_keys
1003,2024-01-01,2026-09-29,1003


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.dim_address
USING DELTA
AS
SELECT
    address_id AS address_key,
    address_id,
    customer_id,
    address_type,
    address_line1,
    address_line2,
    city,
    state,
    postal_code,
    country,
    is_default,
    is_effective_default,
    created_at,
    updated_at
FROM ecommerce.silver.customer_addresses;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT address_id) AS distinct_addresses,
    COUNT_IF(address_id IS NULL) AS null_address_ids
FROM ecommerce.gold.dim_address;

total_rows,distinct_addresses,null_address_ids
30000,30000,0


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fact_orders
USING DELTA
AS
SELECT
    o.order_id,
    c.customer_key,
    d.date_key AS order_date_key,
    a.address_key AS shipping_address_key,

    o.order_status,
    o.payment_status,
    o.shipping_status,

    o.total_amount,
    o.discount_amount,
    o.tax_amount,
    o.final_amount,

    o.created_at,
    o.updated_at

FROM ecommerce.silver.orders o

LEFT JOIN ecommerce.gold.dim_customer c
    ON o.customer_id = c.customer_id

LEFT JOIN ecommerce.gold.dim_date d
    ON TO_DATE(o.order_date) = d.calendar_date

LEFT JOIN ecommerce.gold.dim_address a
    ON o.shipping_address_id = a.address_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS distinct_orders,
    COUNT_IF(customer_key IS NULL) AS missing_customer_keys,
    COUNT_IF(order_date_key IS NULL) AS missing_date_keys,
    COUNT_IF(shipping_address_key IS NULL) AS missing_shipping_address_keys,
    ROUND(SUM(final_amount), 2) AS total_final_amount
FROM ecommerce.gold.fact_orders;

total_rows,distinct_orders,missing_customer_keys,missing_date_keys,missing_shipping_address_keys,total_final_amount
100000,100000,0,0,0,5338856525.64


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fact_order_items
USING DELTA
AS
SELECT
    oi.order_item_id,
    oi.order_id,
    fo.customer_key,
    fo.order_date_key,

    COALESCE(dp.product_key, 0) AS product_key,
    oi.product_id AS source_product_id,

    oi.quantity,
    oi.unit_price,
    oi.discount_amount,
    oi.tax_amount,
    oi.line_total,

    oi.created_at,
    oi.updated_at

FROM ecommerce.silver.order_items oi

LEFT JOIN ecommerce.gold.fact_orders fo
    ON oi.order_id = fo.order_id

LEFT JOIN ecommerce.gold.dim_product dp
    ON oi.product_id = dp.product_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_item_id) AS distinct_order_items,
    COUNT_IF(customer_key IS NULL) AS missing_customer_keys,
    COUNT_IF(order_date_key IS NULL) AS missing_date_keys,
    COUNT_IF(product_key IS NULL) AS missing_product_keys,
    COUNT_IF(product_key = 0) AS unknown_product_items,
    ROUND(SUM(line_total), 2) AS total_line_amount
FROM ecommerce.gold.fact_order_items;

total_rows,distinct_order_items,missing_customer_keys,missing_date_keys,missing_product_keys,unknown_product_items,total_line_amount
299997,299997,0,0,0,57,34997976402.09


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fact_payments
USING DELTA
AS
SELECT
    p.payment_id,
    p.order_id,
    fo.customer_key,
    fo.order_date_key,
    p.payment_method,
    p.transaction_id,
    p.payment_amount,
    p.payment_status,
    CAST(DATE_FORMAT(TO_DATE(p.payment_date), 'yyyyMMdd') AS INT)
        AS payment_date_key,
    p.refund_amount,
    p.payment_date,
    p.created_at,
    p.updated_at
FROM ecommerce.silver.payments p
LEFT JOIN ecommerce.gold.fact_orders fo
    ON p.order_id = fo.order_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT payment_id) AS distinct_payments,
    COUNT_IF(customer_key IS NULL) AS missing_customer_keys,
    COUNT_IF(order_date_key IS NULL) AS missing_order_date_keys,
    COUNT_IF(payment_date_key IS NULL) AS missing_payment_date_keys,
    ROUND(SUM(payment_amount), 2) AS total_payment_amount,
    ROUND(SUM(refund_amount), 2) AS total_refund_amount
FROM ecommerce.gold.fact_payments;

total_rows,distinct_payments,missing_customer_keys,missing_order_date_keys,missing_payment_date_keys,total_payment_amount,total_refund_amount
100000,100000,0,0,0,4528949067.34,199203586.28


In [0]:
%sql
SELECT 'dim_customer' AS table_name, COUNT(*) AS total_rows
FROM ecommerce.gold.dim_customer

UNION ALL

SELECT 'dim_product', COUNT(*)
FROM ecommerce.gold.dim_product

UNION ALL

SELECT 'dim_date', COUNT(*)
FROM ecommerce.gold.dim_date

UNION ALL

SELECT 'dim_address', COUNT(*)
FROM ecommerce.gold.dim_address

UNION ALL

SELECT 'fact_orders', COUNT(*)
FROM ecommerce.gold.fact_orders

UNION ALL

SELECT 'fact_order_items', COUNT(*)
FROM ecommerce.gold.fact_order_items

UNION ALL

SELECT 'fact_payments', COUNT(*)
FROM ecommerce.gold.fact_payments;

table_name,total_rows
dim_customer,25001
dim_product,5000
dim_date,1003
dim_address,30000
fact_orders,100000
fact_order_items,299997
fact_payments,100000


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS orders_with_payments,
    SUM(CASE WHEN payment_count = 1 THEN 1 ELSE 0 END)
        AS orders_with_one_payment,
    SUM(CASE WHEN payment_count > 1 THEN 1 ELSE 0 END)
        AS orders_with_multiple_payments
FROM (
    SELECT
        order_id,
        COUNT(*) AS payment_count
    FROM ecommerce.gold.fact_payments
    GROUP BY order_id
) p;

total_orders,orders_with_payments,orders_with_one_payment,orders_with_multiple_payments
63221,63221,36807,26414


In [0]:
%sql
SELECT
    payment_count,
    COUNT(*) AS number_of_orders
FROM (
    SELECT
        order_id,
        COUNT(*) AS payment_count
    FROM ecommerce.gold.fact_payments
    GROUP BY order_id
) p
GROUP BY payment_count
ORDER BY payment_count;

payment_count,number_of_orders
1,36807
2,18410
3,6061
4,1592
5,290
6,56
7,4
8,1


In [0]:
%sql
WITH payment_summary AS (
    SELECT
        order_id,
        SUM(payment_amount) AS total_paid,
        SUM(refund_amount) AS total_refunded,
        COUNT(*) AS payment_count
    FROM ecommerce.gold.fact_payments
    GROUP BY order_id
)
SELECT
    COUNT(*) AS orders_with_payment_records,
    SUM(CASE
        WHEN ABS(COALESCE(p.total_paid, 0) - o.final_amount) < 0.01
        THEN 1 ELSE 0
    END) AS orders_where_payments_equal_final_amount,
    SUM(CASE
        WHEN COALESCE(p.total_paid, 0) < o.final_amount
        THEN 1 ELSE 0
    END) AS orders_with_payments_below_final_amount,
    SUM(CASE
        WHEN COALESCE(p.total_paid, 0) > o.final_amount
        THEN 1 ELSE 0
    END) AS orders_with_payments_above_final_amount,
    ROUND(SUM(p.total_paid), 2) AS total_recorded_payments,
    ROUND(SUM(p.total_refunded), 2) AS total_recorded_refunds
FROM ecommerce.gold.fact_orders o
JOIN payment_summary p
    ON o.order_id = p.order_id;

orders_with_payment_records,orders_where_payments_equal_final_amount,orders_with_payments_below_final_amount,orders_with_payments_above_final_amount,total_recorded_payments,total_recorded_refunds
63221,36250,5998,20973,4528949067.34,199203586.28


In [0]:
%sql
SELECT
    payment_status,
    COUNT(*) AS payment_records,
    COUNT(DISTINCT order_id) AS affected_orders,
    ROUND(SUM(payment_amount), 2) AS payment_amount,
    ROUND(SUM(refund_amount), 2) AS refund_amount
FROM ecommerce.gold.fact_payments
GROUP BY payment_status
ORDER BY payment_records DESC;

payment_status,payment_records,affected_orders,payment_amount,refund_amount
Success,79966,55041,4264327178.19,0.00
Failed,8079,7750,0.00,0.00
Pending,6966,6719,0.00,0.00
Refunded,4989,4865,264621889.15,199203586.28


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.payment_status_summary
USING DELTA
AS
SELECT
    payment_status,
    COUNT(*) AS payment_records,
    COUNT(DISTINCT order_id) AS affected_orders,
    SUM(payment_amount) AS total_payment_amount,
    SUM(refund_amount) AS total_refund_amount,
    CURRENT_TIMESTAMP() AS summary_created_at
FROM ecommerce.gold.fact_payments
GROUP BY payment_status;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM ecommerce.gold.payment_status_summary
ORDER BY payment_records DESC;

payment_status,payment_records,affected_orders,total_payment_amount,total_refund_amount,summary_created_at
Success,79966,55041,4264327178.19,0.00,2026-10-11T06:59:32.944Z
Failed,8079,7750,0.00,0.00,2026-10-11T06:59:32.944Z
Pending,6966,6719,0.00,0.00,2026-10-11T06:59:32.944Z
Refunded,4989,4865,264621889.15,199203586.28,2026-10-11T06:59:32.944Z


In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM ecommerce.silver.order_items) AS silver_records,

    (SELECT COUNT(*)
     FROM ecommerce.gold.fact_order_items) AS gold_fact_records,

    (SELECT COUNT(*)
     FROM ecommerce.silver.order_item_quarantine) AS quarantined_records,

    (SELECT COUNT(*)
     FROM ecommerce.gold.fact_order_items
     WHERE order_item_id IN (1001, 1002, 1003)) AS quarantined_items_in_gold;

silver_records,gold_fact_records,quarantined_records,quarantined_items_in_gold
299997,299997,3,0


In [0]:
tables = [
    "dim_customer",
    "dim_product",
    "dim_date",
    "dim_address",
    "fact_orders",
    "fact_order_items",
    "fact_payments",
    "payment_status_summary"
]

base_path = "abfss://gold@stcommercesrc.dfs.core.windows.net/dimensional_model"

for table_name in tables:
    source_table = f"ecommerce.gold.{table_name}"
    target_path = f"{base_path}/{table_name}"

    (
        spark.table(source_table)
        .write
        .format("parquet")
        .mode("overwrite")
        .save(target_path)
    )

    print(f"Export completed: {table_name}")

Export completed: dim_customer
Export completed: dim_product
Export completed: dim_date
Export completed: dim_address
Export completed: fact_orders
Export completed: fact_order_items
Export completed: fact_payments
Export completed: payment_status_summary
